In [1]:
import boto3
print(f"Boto3 version: {boto3.__version__}")


Boto3 version: 1.43.89


In [2]:
usage_history = []

In [3]:
import boto3

region = "ap-south-1"
target_model_id = "anthropic.claude-haiku-4-5-20251001-v1:0"
# initialize a boto3 session with the specified region
session = boto3.Session(region_name=region)

bedrock = session.client("bedrock")
# list all inference profiles and filter for the one that matches the target model ID
profiles = bedrock.list_inference_profiles(
    typeEquals="SYSTEM_DEFINED"
)["inferenceProfileSummaries"]

matching_profiles = [
    profile
    for profile in profiles
    if any(
        target_model_id in model.get("modelArn", "")
        for model in profile.get("models", [])
    )
]

if not matching_profiles:
    raise RuntimeError(
        f"No inference profile for {target_model_id} is available in {region}."
    )

model_id = matching_profiles[0]["inferenceProfileId"]
print(f"Using inference profile: {model_id}")



Using inference profile: global.anthropic.claude-haiku-4-5-20251001-v1:0


**claude do not store any messages. SO, for multi turn chat type conversation , the messages have to be maintained/storted locally. Provide this list with every follow up request
**

In [4]:
client = session.client("bedrock-runtime")

In [5]:
def add_user_message(messages, text):

    user_message = {
        "role": "user",
        "content": text,
    }
    messages.append(user_message)
    return user_message

def add_assistant_message(messages, text):

    assistant_message = {
        "role": "assistant",
        "content": text,
    }
    messages.append(assistant_message)
    return assistant_message

def chat_with_model(messages):
    response = client.converse(
        modelId=model_id,
        messages=messages,
    )
    return response["output"]["message"]["content"][0]["text"]


In [6]:
def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "content": [{"text": text}],
    })


def add_assistant_message(messages, text):
    messages.append({
        "role": "assistant",
        "content": [{"text": text}],
    })


def chat_with_model(messages):
    # print(f"Messages being sent to model: {messages}")
    response = client.converse(
        modelId=model_id,
        messages=messages,
    )
    return response["output"]["message"]["content"][0]["text"]


# Make a starting list of messages.
messages = []

# Add the initial user question.
add_user_message(messages, "What's 1+1?")
answer = chat_with_model(messages)
print(answer)

# Add the assistant answer before sending the follow-up.
add_assistant_message(messages, answer)
add_user_message(messages, "And 3 more added to that?")

answer = chat_with_model(messages)
print(answer)

1 + 1 = 2
2 + 3 = 5


## Chat bot excercise

In [7]:
# input_text = |

In [11]:
messages=[]
user_text = input("You: ").strip()
print(f"User input: {user_text}")



User input: tell me something about Ai in 4 lines


In [ ]:
while user_text and user_text != "exit":
    add_user_message(messages, user_text)
    answer = chat_with_model(messages)
    print(f"Assistant: {answer}")
    add_assistant_message(messages, answer)
    user_text = input("You: ").strip()
    print(f"User input: {user_text}")
else:
    print("user endede the conversation.")

Assistant: # AI: A Quick Overview

Artificial Intelligence enables computers to learn from data and make decisions without explicit programming. Modern AI powers everything from smartphone assistants to medical diagnosis and autonomous vehicles. Machine learning, a core AI technique, improves performance through experience rather than following preset rules. As AI advances, it's reshaping industries while raising important questions about ethics, bias, and human oversight.
User input: ok, what about AI taking job from ppl
Assistant: # AI and Jobs: The Real Picture

AI is automating routine tasks like data entry and customer service, which will displace some workers in certain industries. However, history shows technology also creates new jobs—think of how computers eliminated some roles but created millions in tech, design, and support. The key challenge is the transition period: workers need retraining programs and time to adapt before new opportunities emerge. Success depends on how 